In [1]:
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [2]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

In [3]:
BATCH_SIZE = 128 * torch.cuda.device_count() 
EPOCHS = 30
LEARNING_RATE = 0.001
NUM_CLASSES = 100
DATA_DIR = "data/cifar100_images"
CSV_PATH = os.path.join(DATA_DIR, "labels.csv")

In [4]:
df = pd.read_csv(CSV_PATH)
train_n_val_df = df[df['split']=='train'].reset_index(drop=True)
test_df = df[df['split']=='test'].reset_index(drop=True)

train_df, val_df = train_test_split(train_n_val_df, test_size=0.1, stratify=train_n_val_df['label_idx'], random_state=42)
train_df.reset_index(drop=True)
val_df.reset_index(drop=True)


,image_path,label_idx,label_name,split
0,train/train_22572.png,2,baby,train
1,train/train_27890.png,92,tulip,train
2,train/train_26500.png,52,oak_tree,train
3,train/train_09256.png,21,chimpanzee,train
4,train/train_18059.png,44,lizard,train
...,...,...,...,...
4995,train/train_41688.png,39,keyboard,train
4996,train/train_41087.png,25,couch,train
4997,train/train_47207.png,65,rabbit,train
4998,train/train_00959.png,75,skunk,train


In [5]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

val_n_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

In [6]:
class CIFAR100Dataset(Dataset):
    def __init__(self, df, base_dir, transform=None):
        self.df = df
        self.base_dir = base_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.base_dir, self.df.iloc[idx]['image_path'])

        image = Image.open(image_path).convert("RGB")
        label = int(self.df.iloc[idx]['label_idx'])

        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = CIFAR100Dataset(train_df, DATA_DIR, train_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

val_dataset = CIFAR100Dataset(val_df, DATA_DIR, val_n_test_transform)
val_loader  = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

test_dataset = CIFAR100Dataset(test_df, DATA_DIR, val_n_test_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

class CIFAR100Net(nn.Module):
    def __init__(self, num_classes=100):
        super(CIFAR100Net, self).__init__()
        self.in_channels = 64
        
        # Khối chuẩn bị ban đầu
        self.prep = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        # Các tầng học đặc trưng (Tăng channels, giảm kích thước ảnh)
        self.layer1 = self._make_layer(64, 2, stride=1)   # 32x32
        self.layer2 = self._make_layer(128, 2, stride=2)  # 16x16
        self.layer3 = self._make_layer(256, 2, stride=2)  # 8x8
        self.layer4 = self._make_layer(512, 2, stride=2)  # 4x4
        
        # Phân loại
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(ResidualBlock(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# Khởi tạo mô hình
model = CIFAR100Net(num_classes=NUM_CLASSES)

# KÍCH HOẠT MULTI-GPU (Dùng cả 2 thẻ T4)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

model = model.to(device)

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Scheduler giảm LR theo hàm Cosine, giúp mô hình hội tụ sâu ở các epoch cuối
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [9]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

print("Bắt đầu huấn luyện...")
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Cập nhật Learning Rate
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Lưu lịch sử
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%")
    
    # Lưu checkpoint tốt nhất
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Nếu dùng DataParallel, cần lưu model.module thay vì model trực tiếp để tránh lỗi khi load lại sau này
        state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(state_dict, 'best_cifar100_model.pth')
        print(f"--> Đã lưu mô hình mới tốt nhất với Val Acc: {best_val_acc:.2f}%")

print("Huấn luyện hoàn tất!")

In [ ]:
# 1. Vẽ đồ thị Loss & Accuracy
epochs_range = range(1, EPOCHS + 1)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss')
plt.plot(epochs_range, history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss History')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_acc'], label='Train Acc')
plt.plot(epochs_range, history['val_acc'], label='Val Acc')
plt.legend()
plt.title('Accuracy History')

plt.show()

# 2. Đánh giá trên tập Test
# Load lại trọng số tốt nhất
best_model = CIFAR100Net(num_classes=NUM_CLASSES).to(device)
best_model.load_state_dict(torch.load('best_cifar100_model.pth'))

test_loss, test_acc = validate(best_model, test_loader, criterion, device)
print(f"\n[KẾT QUẢ CUỐI CÙNG] Đồ chính xác trên tập TEST: {test_acc:.2f}%")